In [1]:
import nest_asyncio
nest_asyncio.apply()  # Cho phép nest event loop trong Jupyter

import sys
sys.path.insert(0, "/home/duoc/workflow/api")  # thư mục chứa folder app/
print("Cấu hình sys.path thành công! Thư mục hiện tại:")
import os
os.chdir("/home/duoc/workflow/api")  # Thêm dòng này TRƯỚC khi import app

Cấu hình sys.path thành công! Thư mục hiện tại:


In [2]:
# ===== MOCK DATASET (rút gọn từ dữ liệu thật) =====

docs = [
    {
        "id": "kw_01",
        "doc_type": "keyword",
        "text": """
        nhà hàng quán nhậu hải sản đà nẵng ngon rẻ
        nhà hàng quán nhậu hải sản đà nẵng gần biển
        nhà hàng quán nhậu hải sản đà nẵng nổi tiếng
        """
    },

    {
        "id": "brand_01",
        "doc_type": "brand",
        "text": """
        Hải sản Mộc quán Đà Nẵng.
        Giờ mở cửa 10:30–23:45.
        Không gian ăn tại chỗ, takeaway, delivery.
        Cơ sở Đà Nẵng và Nha Trang.
        MICHELIN selected.
        """
    },

    {
        "id": "post_01",
        "doc_type": "social_post",
        "text": """
        THƯỞNG THỨC HẢI SẢN CHUẨN VỊ - SĂN VIEW PHÁO HOA DIFF.
        Đặt bàn trước.
        Không gian chill.
        Hải sản phục vụ nhóm bạn và gia đình.
        """
    },

    {
        "id": "comment_01",
        "doc_type": "comment",
        "text": """
        Quá tuyệt vời.
        Cho em đặt bàn đêm 27/6.
        """
    }
]

In [ ]:
# CELL — TEST RAG WITH REAL RESEARCH DATA

import asyncio
from PIL import Image

from app.rag.service_marketing import (
    RAG,
    MarketingRAG,
    ImageRAG,
)


BUSINESS_ID="4a484e79-605d-4c58-b45c-40089b49a1b7"


async def test():

    print("=== INIT ===")

    rag=RAG()


    # -------------------------
    # 1 KEYWORD RAG
    # -------------------------

    print("\n=== ADD KEYWORDS ===")

    keyword_doc="""
    nhà hàng quán nhậu hải sản đà nẵng ngon rẻ
    nhà hàng quán nhậu hải sản đà nẵng gần biển
    nhà hàng quán nhậu hải sản đà nẵng nổi tiếng
    nhà hàng quán nhậu hải sản đà nẵng review
    """

    await rag.add(
        keyword_doc,
        business_id=BUSINESS_ID,
        document_type="keyword",
        source_id="research_keyword",
    )


    # -------------------------
    # 2 BRAND
    # -------------------------

    print("\n=== ADD BRAND ===")

    brand_doc="""
    Hải sản Mộc quán Đà Nẵng.

    Giờ mở cửa:
    10:30–23:45

    Đà Nẵng:
    26 Tô Hiến Thành

    Hải sản.
    Delivery.
    Takeaway.

    Michelin selected.
    """

    await rag.add(
        brand_doc,
        business_id=BUSINESS_ID,
        document_type="brand_guideline",
        source_id="facebook_intro",
    )


    # -------------------------
    # 3 SOCIAL
    # -------------------------

    print("\n=== ADD POSTS ===")

    social_doc="""
    THƯỞNG THỨC HẢI SẢN CHUẨN VỊ.

    Săn view pháo hoa DIFF.

    Không gian chill.

    Đặt bàn trước.
    """

    await rag.add(
        social_doc,
        business_id=BUSINESS_ID,
        document_type="social_post",
        source_id="fb_post_01",
    )


    # -------------------------
    # 4 COMMENT
    # -------------------------

    print("\n=== ADD COMMENT ===")

    comment_doc="""
    Quá tuyệt vời.

    Cho em đặt bàn đêm 27/6.
    """

    await rag.add(
        comment_doc,
        business_id=BUSINESS_ID,
        document_type="comment",
        source_id="comment_01",
    )



    # ==================================================
    # SEARCH
    # ==================================================

    TESTS=[

        "món chủ lực của mộc",

        "khách hàng muốn trải nghiệm gì",

        "seo nhà hàng",

        "đặt bàn xem pháo hoa",

        "điểm mạnh thương hiệu",

    ]


    for q in TESTS:

        print("\n")
        print("="*70)
        print("QUERY:",q)

        r=await rag.search(
            query=q,
            business_id=BUSINESS_ID,
            top_k=5,
        )

        print("SOURCE:",r.source)

        for i,c in enumerate(r.chunks,1):

            print(
                f"\n[{i}]",
                round(c.score,3),
                c.meta.get("document_type"),
                c.meta.get("source_id"),
            )

            print(
                c.text[:180]
            )


    # ==================================================
    # MARKETING SEARCH
    # ==================================================

    print("\n=== MARKETING ===")

    mrag=MarketingRAG(rag)

    r=await mrag.search_for_content(
        query="viết post quảng bá hải sản cho gia đình",
        business_id=BUSINESS_ID,
        content_type="social",
    )

    for x in r.chunks:

        print(
            x.meta.get(
                "document_type"
            )
        )



    # ==================================================
    # IMAGE
    # ==================================================

    print("\n=== IMAGE ===")

    irag=ImageRAG()

    img=Image.new(
        "RGB",
        (224,224),
        "red",
    )

    await irag.add(
        img,
        image_id="menu_mock",
        business_id=BUSINESS_ID,
    )

    rs=await irag.search(img)

    print(rs)



    # ==================================================
    # STATS
    # ==================================================

    print("\n=== STATS ===")

    print(
        rag.stats()
    )

    print(
        irag.stats()
    )



await test()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# extract_brand_voice — PHIÊN BẢN MỚI (dùng slicer 4-LLM song song)
# Input: data thô (task/result/posts/comments/events) — KHÔNG còn nhận
#        brand_voice_input của signal_extractor nữa.
# Output: vẫn là dict khớp BrandEightFields (qua _sanitize_raw/_validate)
#         như cũ, để không phải sửa 2 background task ở phần unpack ORM.
# ═══════════════════════════════════════════════════════════════════

import asyncio
import re
from datetime import datetime, timezone
from typing import Any, Dict, List

from app.llm_clients import call_groq
# _sanitize_raw, _validate, BrandEightFields: giữ nguyên import cũ của bạn
# from app.brand_voice.schema import _sanitize_raw, _validate


# ───────────────────────────────────────────────────────────────────
# SLICER — chia posts/comments thành 4 lát input cho 4 "kiến"
# ───────────────────────────────────────────────────────────────────

def _k1_posts(posts: List[dict]) -> List[dict]:
    return sorted(posts, key=lambda p: len(p.get("content", "")), reverse=True)[:5]


def _k2_posts(posts: List[dict]) -> List[dict]:
    def cls(p):
        c = p.get("content", "").lower()
        if "michelin" in c:
            return 0
        if any(x in c for x in ["diff", "pháo hoa", "sự kiện"]):
            return 1
        if any(x in c for x in ["giao hàng", "tận nhà", "delivery", "take-away"]):
            return 2
        return 3

    buckets: List[List[dict]] = [[], [], [], []]
    for p in posts:
        buckets[cls(p)].append(p)

    r: List[dict] = []
    for bucket in buckets:
        r.extend(bucket[:2])
    if len(r) < 8:
        used = {p["id"] for p in r}
        for p in posts:
            if p["id"] not in used:
                r.append(p)
            if len(r) >= 8:
                break
    return r[:8]


def _k3_posts(posts: List[dict]):
    minimal = []
    for p in posts:
        c = p.get("content", "")
        s = [x.strip() for x in c.replace("!", ".").replace("?", ".").split(".") if x.strip()]
        minimal.append({
            "id": p["id"],
            "first": s[0][:100] if s else "",
            "last": s[-1][:100] if len(s) > 1 else "",
            "wc": len(c.split()),
            "emoji": any(ord(ch) > 10000 for ch in c),
            "hash": "#" in c,
            "bi": "----------" in c,
            "bull": c.count("\n-"),
        })
    return minimal, posts[:2]


def _k4_posts(posts: List[dict]) -> List[dict]:
    m = ["Hotline:", "CS1", "Đặt bàn", "Inbox", "Gọi điện"]
    r = [p for p in posts if any(x in p.get("content", "") for x in m)]
    return r if len(r) >= 3 else posts[-4:]


def prepare_kien_inputs(data: Dict[str, Any]) -> Dict[str, Any]:
    posts = data.get("posts", [])
    comments = data.get("comments", [])
    serp = data.get("result", {}).get("serp_data", {})
    fb = data.get("result", {}).get("fb_brand", {})
    mc = [c for c in comments if len(c.get("comment", "")) > 20 and c.get("author") != "Ẩn danh"]
    m3, f3 = _k3_posts(posts)
    return {
        "k1": {
            "posts": _k1_posts(posts),
            "fb": fb.get("intro", "")[:2000],
            "name": data.get("task", {}).get("business_name", ""),
        },
        "k2": {
            "posts": _k2_posts(posts),
            "comments": mc[:15],
            "name": data.get("task", {}).get("business_name", ""),
        },
        "k3": {
            "minimal": m3,
            "full": f3,
            "kc": serp.get("keyword_cluster", []),
            "cp": serp.get("competitor_pattern", []),
        },
        "k4": {
            "posts": _k4_posts(posts),
            "bc": [
                c for c in mc
                if any(x in c.get("comment", "").lower() for x in ["đặt bàn", "menu", "hotline", "giá"])
            ][:10],
        },
    }


# ───────────────────────────────────────────────────────────────────
# PROMPT BUILDERS
# ───────────────────────────────────────────────────────────────────

def _pk1(i: Dict[str, Any]) -> str:
    pt = "\n\n---\n".join([f"ID:{p['id']}\n{p.get('content', '')[:3000]}" for p in i["posts"]])
    return f"""Bạn là chuyên gia brand DNA. Đọc bài viết và viết PHÂN TÍCH TỰ NHIÊN về LINH HỒN thương hiệu {i["name"]}.

FB INTRO: {i["fb"][:1500]}

BÀI VIẾT:
{pt}

Viết 1 đoạn văn liền mạch 200-400 từ: brand tự hào điều gì NHẤT? Không bao giờ làm gì vì tiền? 3 metaphor tự mô tả? Khác biệt đối thủ?

KHÔNG dùng "thân thiện", "chuyên nghiệp", "chất lượng".

QUAN TRỌNG — KẾT THÚC bằng EXACTLY 5 dòng, mỗi dòng 1 SIGNATURE PHRASE trong ngoặc kép:
"phrase 1"
"phrase 2"
"phrase 3"
"phrase 4"
"phrase 5"

Các phrase này PHẢI là những câu brand DÙNG NHIỀU LẦN trong bài viết trên."""


def _pk2(i: Dict[str, Any]) -> str:
    pt = "\n\n---\n".join([f"[{p['id']}]\n{p.get('content', '')[:2500]}" for p in i["posts"]])
    ct = "\n".join([f"- {c['author']}: {c.get('comment', '')[:200]}" for c in i["comments"]])
    return f"""Bạn là linguist phân tích tone & style của {i["name"]}. Đọc bài viết và comments, viết PHÂN TÍCH TỰ NHIÊN.

BÀI VIẾT:
{pt}

COMMENTS:
{ct}

Viết 1 đoạn 200-400 từ: emotional arc? Tone chính (CHỌN 2-3 TỪ CỤ THỂ)? Pronoun pattern? Bilingual strategy? 4 sliders (0-100): funny-serious, formal-casual, respectful-irreverent, enthusiastic-matter_of_fact.

KHÔNG JSON. Viết như báo cáo phân tích."""


def _pk3(i: Dict[str, Any]) -> str:
    mt = "\n".join([
        f"P{m['id']}: first='{m['first']}' last='{m['last']}' wc={m['wc']} emo={m['emoji']} hash={m['hash']} bi={m['bi']} bull={m['bull']}"
        for m in i["minimal"]
    ])
    ft = "\n\n---\n".join([p.get("content", "")[:2000] for p in i["full"]])
    return f"""Bạn là data analyst. Đọc metadata và trả lời BẰNG SỐ LIỆU.

META 11 POSTS:
{mt}

FULL SAMPLES:
{ft}

KEYWORDS: {i['kc']}

YÊU CẦU — Trả lời CHÍNH XÁC bằng số liệu từ metadata:
1. Bao nhiêu post có emoji? (đếm từ cột emo=true)
2. Bao nhiêu post có hashtag? (đếm từ cột hash=true)
3. Hashtag ở đầu hay cuối post?
4. Bao nhiêu post có bullet points? (đếm từ cột bull>0)
5. Bullet style là gì? (dựa vào first/last line)
6. Top 10 phrases brand DÙNG NHIỀU NHẤT (từ full samples, đếm lặp lại)
7. Top 5 words KHÔNG BAO GIỜ dùng (so với keywords ngành)
8. 5 SIGNATURE PHRASES — đặt trong ngoặc kép, mỗi phrase 1 dòng

KHÔNG phân tích. KHÔNG suy đoán. Chỉ trả lời 8 câu trên."""


def _pk4(i: Dict[str, Any]) -> str:
    pt = "\n\n---\n".join([f"[{p['id']}]\n{p.get('content', '')[-1500:]}" for p in i["posts"]])
    bt = "\n".join([f"- {c['author']}: {c.get('comment', '')[:150]}" for c in i["bc"]])
    return f"""Bạn là conversion strategist phân tích CTA.

CTA SECTIONS:
{pt}

BOOKING COMMENTS:
{bt}

Viết 1 đoạn 150-300 từ: CTA style direct/soft? Top 5 phrases? Urgency framing? Conversion triggers? CTA components? Response pattern?

KHÔNG JSON. Audit report."""


# ───────────────────────────────────────────────────────────────────
# 4 GỌI GROQ SONG SONG
# ⚠️ Giả định call_groq() là async (khớp docstring cũ "Gọi Groq async").
#    Nếu call_groq() trong app/llm_clients.py là hàm SYNC (blocking),
#    bỏ chữ `await` ở 4 dòng dưới — nếu không sẽ lỗi "object is not awaitable".
# ───────────────────────────────────────────────────────────────────

async def _k1(i: Dict[str, Any]) -> str:
    return await call_groq(_pk1(i), 1500)


async def _k2(i: Dict[str, Any]) -> str:
    return await call_groq(_pk2(i), 1500)


async def _k3(i: Dict[str, Any]) -> str:
    return await call_groq(_pk3(i), 1500)


async def _k4(i: Dict[str, Any]) -> str:
    return await call_groq(_pk4(i), 1200)


async def run_kiens(data: Dict[str, Any]) -> Dict[str, str]:
    inp = prepare_kien_inputs(data)
    logger.info(
        "run_kiens | K1=%d K2=%d+%d K3=%d+%d K4=%d+%d",
        len(inp["k1"]["posts"]),
        len(inp["k2"]["posts"]), len(inp["k2"]["comments"]),
        len(inp["k3"]["minimal"]), len(inp["k3"]["full"]),
        len(inp["k4"]["posts"]), len(inp["k4"]["bc"]),
    )
    r = await asyncio.gather(
        asyncio.create_task(_k1(inp["k1"])),
        asyncio.create_task(_k2(inp["k2"])),
        asyncio.create_task(_k3(inp["k3"])),
        asyncio.create_task(_k4(inp["k4"])),
    )
    return {"k1": r[0], "k2": r[1], "k3": r[2], "k4": r[3]}


# ───────────────────────────────────────────────────────────────────
# AGGREGATOR HELPERS — string utils & extractors
# ───────────────────────────────────────────────────────────────────

def _between(t, s, e=None):
    try:
        i = t.lower().find(s.lower())
        if i == -1:
            return ""
        i += len(s)
        if e:
            j = t.lower().find(e.lower(), i)
            return t[i:j].strip() if j != -1 else t[i:].strip()
        return t[i:].strip()
    except Exception:
        return ""


def _slider(t, lo, hi):
    t = t.lower()
    lc, hc = t.count(lo), t.count(hi)
    tot = lc + hc
    return 50 if tot == 0 else int((hc / tot) * 100)


def _pronouns(t):
    t = t.lower()
    r = {"ai": "Chúng tôi", "reader": "Bạn"}
    if t.count("nhà mộc") > 2:
        r["brand_ref"] = "nhà Mộc"
    elif t.count("mộc quán") > 2:
        r["brand_ref"] = "Mộc Quán"
    return r


def _tone_base(t):
    t = t.lower()
    sp = ["proud", "humble", "warm", "precise", "grateful", "excited", "passionate", "earnest", "dignified"]
    found = [w for w in sp if w in t]
    if found:
        return [w.title() for w in found[:3]]
    r = []
    if any(x in t for x in ["michelin", "vinh danh", "tự hào", "vinh dự", "ngôi sao", "bệ phóng", "đích đến"]):
        r.append("Proud")
    if any(x in t for x in ["humble", "khiêm tốn", "tri ân", "cảm ơn", "gật đầu"]):
        r.append("Humble")
    if any(x in t for x in ["warm", "ấm cúng", "chào đón", "nhà mộc"]):
        r.append("Warm")
    if any(x in t for x in ["grateful", "tri ân", "cảm ơn sâu sắc", "đồng hành"]):
        r.append("Grateful")
    if any(x in t for x in ["precise", "chính xác", "chi tiết", "tuyệt đối"]):
        r.append("Precise")
    if any(x in t for x in ["excited", "hào hứng", "bùng nổ", "rộn ràng"]):
        r.append("Excited")
    return r if r else ["Earnest", "Warm"]


def _sig_phrases(k1, k3):
    k1_lines = [l.strip() for l in k1.strip().split("\n") if l.strip().startswith('"') and l.strip().endswith('"')]
    q = [l.strip('"') for l in k1_lines if len(l.strip('"')) > 10]
    q += re.findall(r'"([^"]{10,80})"', k3)
    sig_sec = _between(k3, "signature", "format") or _between(k3, "8.", "\n\n") or k3[-800:]
    q += re.findall(r'"([^"]{10,80})"', sig_sec)
    seen, uniq = set(), []
    for p in q:
        cl = p.strip().lower()
        if cl not in seen and len(p) > 15:
            seen.add(cl)
            uniq.append(p.strip())
    return uniq[:5]


def _format_rules(k3):
    t = k3.lower()
    emo_match = re.search(r'(\d+)\s*post.*emoji', t)
    emo_count = int(emo_match.group(1)) if emo_match else t.count("emoji")
    hash_match = re.search(r'(\d+)\s*post.*hash', t)
    hash_count = int(hash_match.group(1)) if hash_match else t.count("hash")
    bull_match = re.search(r'(\d+)\s*post.*bull', t)
    bull_count = int(bull_match.group(1)) if bull_match else t.count("bull")
    return {
        "paragraphMaxSentences": 5,
        "useEmoji": emo_count > 0,
        "useHashtags": hash_count > 0,
        "bulletPointStyle": "dash" if bull_count > 0 or "dash" in t or "-" in t else "none",
    }


def _avoid(k3):
    s = _between(k3, "không bao giờ", "signature") or _between(k3, "avoid", "signature")
    w = []
    for l in (s or "").split("\n"):
        c = l.strip("- •*123456789. \"'").strip()
        if c and len(c) < 30 and len(c) > 2 and not c.startswith((":", ",", "!", "?")):
            w.append(c)
    return w


def _cta_phrases(k4):
    r = []
    for line in k4.split("\n"):
        line = line.strip()
        if any(m in line.lower() for m in ["đặt bàn", "hotline", "inbox", "gọi điện"]):
            c = line.strip("- •*123456789. ").strip()
            if c and len(c) < 60:
                r.append(c)
    return r[:5] if r else ["Đặt bàn ngay", "Gọi Hotline để giữ chỗ", "Inbox Fanpage"]


def aggregate(k1: str, k2: str, k3: str, k4: str, bid: str, bname: str) -> Dict[str, Any]:
    now = datetime.now(timezone.utc).isoformat()
    sig = _sig_phrases(k1, k3)
    personality = " ".join([p.strip() for p in k1.strip().split("\n\n") if len(p.strip()) > 50][:2])[:800]
    return {
        "business_id": bid,
        "name": bname,
        "personality": personality,
        "tone": {"base": _tone_base(k2), "overrides": {"blog_web": _tone_base(k2)}},
        "style": {"sentenceLength": "medium", "voice": "active", "perspective": "first", "pronouns": _pronouns(k2)},
        "vocabulary": {
            "wordsToUse": sig[:3] if sig else [],
            "wordsToAvoid": _avoid(k3),
            "phrasesToUse": sig[:5],
            "phrasesToAvoid": [],
            "topicsToAvoid": [],
        },
        "format_rules": _format_rules(k3),
        "cta_style": {
            "style": "direct" if any(x in k4.lower() for x in ["direct", "mạnh mẽ", "ngay"]) else "mixed",
            "phrases": _cta_phrases(k4),
        },
        "examples": [{
            "input": f"Tìm kiếm thông tin về {bname}",
            "output": f"{sig[0] if sig else ''}",
            "contentType": "blog_web",
        }],
        "tone_funny_serious": _slider(k2, "funny", "serious"),
        "tone_formal_casual": _slider(k2, "formal", "casual"),
        "tone_respectful_irreverent": _slider(k2, "irreverent", "respectful"),
        "tone_enthusiastic_matter_of_fact": _slider(k2, "matter of fact", "enthusiastic"),
        "created_at": now,
        "updated_at": now,
    }


# ───────────────────────────────────────────────────────────────────
# MAIN FUNCTION — signature mới: nhận data thô (task/result/posts/comments)
# ───────────────────────────────────────────────────────────────────

async def extract_brand_voice(data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Slicer pipeline: chạy 4 Groq call song song (k1-k4) trên posts/comments/
    serp_data/fb_brand, rồi aggregate kết quả text → dict khớp BrandEightFields.

    Args:
        data: {
            task: {business_id, business_name},
            result: {serp_data, fb_brand},
            posts: [...],
            comments: [...],
        }

    Returns:
        Dict khớp BrandEightFields — sẵn sàng unpack vào Brand ORM.

    Raises:
        RuntimeError: 1 trong 4 Groq call thất bại
        ValueError:   Aggregate output không đúng schema
    """
    business_id = data.get("task", {}).get("business_id", "")
    business_name = data.get("task", {}).get("business_name", "")
    logger.info("extract_brand_voice | business=%s", business_name)

    try:
        ko = await run_kiens(data)
    except Exception as exc:
        logger.error("Groq slicer call failed: %s", exc)
        raise RuntimeError(f"Lỗi kết nối Groq: {exc}") from exc

    raw = aggregate(ko["k1"], ko["k2"], ko["k3"], ko["k4"], business_id, business_name)

    try:
        raw = _sanitize_raw(raw)
        validated = _validate(raw)
    except Exception as exc:
        logger.error("Aggregate output không đúng schema: %s", exc)
        raise ValueError(f"Aggregate output không đúng schema: {exc}") from exc

    return validated.model_dump()


# ═══════════════════════════════════════════════════════════════════
# ADAPTER — research_json (đầu ra build_research_json) → data cho slicer
# Dùng để KHÔNG cần gọi extract_brand_signals() nữa.
# ═══════════════════════════════════════════════════════════════════

def _research_json_to_slicer_data(research_json: Dict[str, Any]) -> Dict[str, Any]:
    bc = research_json.get("business_context", {})
    ebv = research_json.get("existing_brand_voice", {})
    return {
        "task": {
            "business_id": bc.get("business_id", ""),
            "business_name": bc.get("business_name", ""),
        },
        "result": {
            "serp_data": research_json.get("market_patterns", {}),
            "fb_brand": ebv.get("fb_brand", {}),
        },
        "posts": ebv.get("posts_raw", []),
        "comments": research_json.get("customer_feedback", {}).get("comments_raw", []),
    }


# ═══════════════════════════════════════════════════════════════════
# 🔧 FIXED BACKGROUND TASK #1 — Extract từ existing research
# ═══════════════════════════════════════════════════════════════════
async def _extract_and_save_bg(
    voice_id: str,
    business_id: str,
    voice_config_dict: Dict[str, Any],
    task_id: int,
) -> None:
    """
    ✅ Case 1: Existing business + existing research
    """
    async with AsyncSessionLocal() as db:
        try:
            business = await _get_business_or_404(business_id, db)
            research = await _get_latest_research(db, business_id)

            if not research:
                raise ValueError("Chưa có research data cho business này. Hãy chạy research trước.")

            fb_posts = await _get_fb_posts_for_research(db, business_id)
            fb_comments = await _get_fb_comments_for_research(db, business_id)

            logger.info(
                "extract_bg | fetched data | business_id=%s | posts=%d | comments=%d",
                business_id, len(fb_posts), len(fb_comments)
            )

            research_json = build_research_json(
                research=research,
                fb_posts=fb_posts,
                fb_comments=fb_comments,
                voice_config=voice_config_dict,
                business=business,
            )

            # ⬇️ THAY ĐỔI: không còn extract_brand_signals() → đổi sang adapter slicer
            slicer_data = _research_json_to_slicer_data(research_json)
            eight_fields = await extract_brand_voice(slicer_data)
            await update_task(db, task_id, steps_done=1)

            result = await db.execute(select(Brand).filter(Brand.id == voice_id))
            voice = result.scalars().first()
            if not voice:
                await fail_task(db, task_id, error_message=f"Brand {voice_id} không tìm thấy.")
                return

            voice.personality = eight_fields["personality"]
            voice.tone = eight_fields["tone"]
            voice.style = eight_fields["style"]
            voice.vocabulary = eight_fields["vocabulary"]
            voice.format_rules = eight_fields["format_rules"]
            voice.cta_style = eight_fields["cta_style"]
            voice.examples = eight_fields.get("examples", [])

            voice.tone_funny_serious = eight_fields.get("tone_funny_serious", 50)
            voice.tone_formal_casual = eight_fields.get("tone_formal_casual", 50)
            voice.tone_respectful_irreverent = eight_fields.get("tone_respectful_irreverent", 50)
            voice.tone_enthusiastic_matter_of_fact = eight_fields.get("tone_enthusiastic_matter_of_fact", 50)

            await db.commit()
            await finish_task(db, task_id, steps_done=2)
            logger.info("extract_bg done | voice_id=%s | task_id=%s", voice_id, task_id)

        except Exception as exc:
            logger.error("extract_bg failed | voice_id=%s | err=%s", voice_id, exc)
            await fail_task(db, task_id, error_message=str(exc))


# ═══════════════════════════════════════════════════════════════════
# 🔧 FIXED BACKGROUND TASK #2 — Research + Extract
# ═══════════════════════════════════════════════════════════════════
async def _research_and_extract_bg(
    voice_id: str,
    business_id: str,
    voice_config_dict: Dict[str, Any],
    task_id: int,
) -> None:
    async with AsyncSessionLocal() as db:
        try:
            business = await _get_business_or_404(business_id, db)
            research = await _get_latest_research(db, business_id)

            if research:
                logger.info("_research_and_extract_bg | Research found in DB. Extracting...")
                fb_posts = await _get_fb_posts_for_research(db, business_id)
                fb_comments = await _get_fb_comments_for_research(db, business_id)
                research_json = build_research_json(
                    research=research,
                    fb_posts=fb_posts,
                    fb_comments=fb_comments,
                    voice_config=voice_config_dict,
                    business=business,
                )
            else:
                logger.info("_research_and_extract_bg | Research NOT found. Running research pipeline...")
                from app.research.research_service import run_research

                query = f"{business.name} {business.address or ''} {business.industry or ''}".strip()
                state = await run_research(
                    db=db,
                    business_id=business.id,
                    query="nhà hàng quán nhậu hải sản Đà Nẵng",
                    fb_url="https://www.facebook.com/mocseafood/",
                    business_name=business.name,
                )

                research_json = {
                    "customer_language": {
                        "suggestions_raw": list(state.suggestions) if hasattr(state, 'suggestions') else state.get("suggestions", []),
                        "suggestions_tagged": state.tagged_suggestions if hasattr(state, 'tagged_suggestions') else state.get("tagged_suggestions", {}),
                    },
                    "market_patterns": state.serp_data if hasattr(state, 'serp_data') else state.get("serp_data", {}),
                    "existing_brand_voice": {
                        "fb_brand": state.fb_data.get("brand", {}) if hasattr(state, 'fb_data') else state.get("fb_data", {}).get("brand", {}),
                        "posts_raw": state.fb_data.get("posts", []) if hasattr(state, 'fb_data') else state.get("fb_data", {}).get("posts", []),
                    },
                    "customer_feedback": {
                        "comments_raw": state.fb_data.get("comments", []) if hasattr(state, 'fb_data') else state.get("fb_data", {}).get("comments", []),
                    },
                    "competitor_insights": {
                        "competitor_pattern": (state.serp_data or {}).get("competitor_pattern", []) if hasattr(state, 'serp_data') else state.get("serp_data", {}).get("competitor_pattern", []),
                    },
                    "business_context": {
                        "business_id": business.id,
                        "business_name": business.name,
                        "industry": getattr(business, "industry", None) or "—",
                        "products": getattr(business, "products", None) or "—",
                        "voice_config": voice_config_dict,
                    },
                }

                business.status = "active"
                await db.commit()

            await update_task(db, task_id, steps_done=1)

            # ⬇️ THAY ĐỔI: không còn extract_brand_signals() → đổi sang adapter slicer
            slicer_data = _research_json_to_slicer_data(research_json)
            eight_fields = await extract_brand_voice(slicer_data)
            await update_task(db, task_id, steps_done=2)

            result = await db.execute(select(Brand).filter(Brand.id == voice_id))
            voice = result.scalars().first()
            if not voice:
                await fail_task(db, task_id, error_message=f"Brand {voice_id} không tìm thấy.")
                return

            fb_brand_info = research_json.get("existing_brand_voice", {}).get("fb_brand", {})
            logo_url = fb_brand_info.get("og_image")
            voice.metadata_info = {
                "logo_url": logo_url,
                "updated_at": _utcnow().isoformat()
            }
            voice.personality = eight_fields["personality"]
            voice.tone = eight_fields["tone"]
            voice.style = eight_fields["style"]
            voice.vocabulary = eight_fields["vocabulary"]
            voice.format_rules = eight_fields["format_rules"]
            voice.cta_style = eight_fields["cta_style"]
            voice.examples = eight_fields.get("examples", [])

            voice.tone_funny_serious = eight_fields.get("tone_funny_serious", 50)
            voice.tone_formal_casual = eight_fields.get("tone_formal_casual", 50)
            voice.tone_respectful_irreverent = eight_fields.get("tone_respectful_irreverent", 50)
            voice.tone_enthusiastic_matter_of_fact = eight_fields.get("tone_enthusiastic_matter_of_fact", 50)

            await db.commit()
            await finish_task(db, task_id, steps_done=3)
            logger.info("_research_and_extract_bg success | voice_id=%s | task_id=%s", voice_id, task_id)

        except Exception as exc:
            logger.error("_research_and_extract_bg failed | voice_id=%s | err=%s", voice_id, exc)
            await fail_task(db, task_id, error_message=str(exc))